# ALBEF 多模态预训练模型代码精读

> **论文**：*Align before Fuse: Vision and Language Representation Learning with Momentum Distillation*（Li et al., NeurIPS 2021）  
> **核心创新**：先用 ITC 对比学习对齐单模态特征（Align），再用 Cross-Attention 融合（Fuse），避免了未对齐特征直接融合导致的次优解。  
> **三大训练任务**：ITC（图文对比）+ ITM（图文匹配）+ MLM（掩码语言模型）

## 一、ALBEF 基础架构（图像编码器 + 文本编码器 + 多模态融合 + 三任务）

本节手动搭建完整 ALBEF 模型，包含：
- `ImageEncoder`：ViT-B/16 骨干网络（与论文完全一致，输出 197 个 token，含 [CLS] + 196 个 patch token）
- `TextEncoder`：完整 12 层 BERT-base 文本编码器
- `MultimodalEncoder`：6 层 MultimodalEncoderLayer（自注意力 + 图文交叉注意力 + FFN）
- `ALBEF`：统一入口，根据 `task` 参数分发到 ITC / ITM / MLM 三个任务

### 1.1 导入依赖

导入 PyTorch、HuggingFace Transformers、torchvision 等所有后续代码依赖的第三方库。

In [ ]:
# ============================================================
# ALBEF 教学版 Notebook - 依赖导入
# 内容：图像编码器 + 文本编码器 + 多模态融合 + 三任务（ITC/ITM/MLM）
# 论文要点：Align Before Fuse（先对比对齐，再 Cross-Attention 融合）
# ============================================================

import torch                              # PyTorch 核心库：张量计算、GPU 加速、自动求导
import torch.nn as nn                     # 神经网络模块：Linear、LayerNorm、Module 等
import torch.nn.functional as F           # 函数式 API：softmax、normalize、cross_entropy 等
from transformers import BertModel, BertTokenizer, BertConfig, ViTModel  # HuggingFace BERT 模型与分词器 + ViT-B/16 图像编码器
import random                             # Python 随机数（演示时可复现）
import numpy as np                        # 数值计算库（设置随机种子）
import math                               # 数学函数（注意力缩放因子 sqrt(d_k)）




### 1.2 图像编码器（ImageEncoder）

以 ViT-B/16 为骨干（与 ALBEF 论文一致），将输入图像切分为 14×14=196 个 patch，加上 [CLS] token 共输出 197 个 token 特征序列。[CLS] token（位于第 0 位）作为全局图像表示用于 ITC 对比学习，其余 196 个 patch token 作为 Key/Value 参与 Cross-Attention 融合。

In [ ]:
class ImageEncoder(nn.Module):
    """
    图像编码器：将输入图像映射为 patch 级 token 特征序列（含 [CLS] token）。
    使用 ViT-B/16（Vision Transformer Base，patch_size=16），与 ALBEF 论文完全一致。
    输出形状：(batch_size, 197, embed_dim)
    其中 197 = 14×14=196 个 patch token + 1 个 [CLS] token（位于序列第 0 位）
    [CLS] token 作为全局图像向量用于 ITC，196 个 patch token 作为 Key/Value 参与 Cross-Attention
    """

    def __init__(self, embed_dim=768, model_name="google/vit-base-patch16-224"):
        """
        初始化图像编码器。
        Args:
            embed_dim (int): 输出特征维度，需与 BERT 隐藏层维度一致，默认 768
            model_name (str): HuggingFace ViT 预训练模型名称，默认 ViT-B/16（224×224 输入）
        """
        super(ImageEncoder, self).__init__()           # 调用父类初始化，注册为 PyTorch 子模块
        self.vit = ViTModel.from_pretrained(model_name,cache_dir="./model_cahce/ViT")  # 加载 HuggingFace 预训练 ViT-B/16 权重
        vit_hidden_size = self.vit.config.hidden_size  # 读取 ViT 的隐藏层维度（ViT-B/16 为 768）
        self.proj = nn.Linear(vit_hidden_size, embed_dim)  # 投影层：vit_hidden_size → embed_dim（B/16 二者均为 768）

    def forward(self, images):
        """
        前向传播：图像 → patch token 特征序列（含 [CLS] token）。
        Args:
            images (Tensor): 形状 (batch_size, 3, 224, 224)，RGB 三通道图像
        Returns:
            features (Tensor): 形状 (batch_size, 197, embed_dim)；
                               197 = 14×14=196 个 patch token + 1 个 [CLS] token；
                               第 0 位为 [CLS] token，用作全局图像向量（ITC 对比学习使用）；
                               第 1~196 位为各 patch 局部视觉特征（Cross-Attention 中作为 Key/Value）
        """
        outputs = self.vit(pixel_values=images)  # ViT-B/16 前向：输入 (B, 3, 224, 224)
        features = outputs.last_hidden_state     # 取最后一层隐藏状态：(B, 197, vit_hidden_size)
        features = self.proj(features)           # 投影到统一维度：(B, 197, embed_dim)
        return features

### 1.3 文本编码器（TextEncoder）

基于 BERT-base 的文本单模态编码器，将 token ID 序列转化为上下文感知的稠密向量。

In [ ]:
class TextEncoder(nn.Module):
    """
    文本编码器：使用完整 BERT-base 将 token 序列编码为上下文向量。
    对应 ALBEF 论文中文本单模态编码器（论文使用 BERT 前 6 层，此处用完整 12 层演示）。
    """

    def __init__(self, embed_dim=768):
        """
        初始化文本编码器。
        Args:
            embed_dim (int): BERT 隐藏层维度，默认 768
        """
        super(TextEncoder, self).__init__()  # 调用父类初始化，注册为 PyTorch 子模块
        self.bert = BertModel.from_pretrained('bert-base-uncased')  # 加载英文 BERT-base（12 层，768 维，12 头）
        self.embed_dim = embed_dim  # 记录嵌入维度，便于后续模块对齐

    def forward(self, input_ids, attention_mask):
        """
        前向传播：token ID 序列 → 上下文向量序列。
        Args:
            input_ids (Tensor): token ID，形状 (batch_size, seq_len)
            attention_mask (Tensor): 1 表示有效 token，0 表示 padding，形状 (batch_size, seq_len)
        Returns:
            last_hidden_state (Tensor): 每个 token 的上下文向量，形状 (batch_size, seq_len, 768)；
                                        第 0 位 [CLS] 可作为句子级别的全局表示，用于 ITC 对比学习
        """
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)  # BERT 12 层 Transformer 前向传播
        return outputs.last_hidden_state  # 取最后一层（第 12 层）隐藏状态作为文本特征

### 1.4 多头交叉注意力（MultiHeadCrossAttention）

图文交叉注意力的核心实现：文本 token 作为 Query，图像 patch 作为 Key/Value，实现视觉信息向文本侧的注入。

In [ ]:
class MultiHeadCrossAttention(nn.Module):
    """
    多头交叉注意力：Query 来自文本，Key/Value 来自图像。
    让文本 token「看到」图像中与之相关的区域，是 ALBEF Fuse 阶段的核心算子。
    """

    def __init__(self, embed_dim=768, num_heads=12):
        """
        初始化多头交叉注意力模块。
        Args:
            embed_dim (int): 输入输出维度，须能被 num_heads 整除，默认 768
            num_heads (int): 注意力头数，BERT-base 默认 12 头
        """
        super(MultiHeadCrossAttention, self).__init__()  # 调用父类初始化，注册多头交叉注意力子模块
        self.embed_dim = embed_dim              # 模型总维度
        self.num_heads = num_heads              # 多头数量
        self.head_dim = embed_dim // num_heads  # 每个头的维度 = 768/12 = 64
        assert self.head_dim * num_heads == embed_dim, "embed_dim must be divisible by num_heads"  # 断言维度可被头数整除，否则无法均匀分配注意力头
        self.q_linear = nn.Linear(embed_dim, embed_dim)  # Query 投影矩阵（来自文本侧）
        self.k_linear = nn.Linear(embed_dim, embed_dim)  # Key 投影矩阵（来自图像侧）
        self.v_linear = nn.Linear(embed_dim, embed_dim)  # Value 投影矩阵（来自图像侧）
        self.out = nn.Linear(embed_dim, embed_dim)       # 多头拼接后的输出投影

    def forward(self, query, key, value, attention_mask=None):
        """
        前向传播：计算文本对图像的多头交叉注意力。
        Args:
            query (Tensor): 文本特征，形状 (batch_size, text_len, embed_dim)
            key (Tensor): 图像特征，形状 (batch_size, img_patches, embed_dim)
            value (Tensor): 图像特征，形状 (batch_size, img_patches, embed_dim)
            attention_mask (Tensor, optional): 文本掩码，形状 (batch_size, text_len)
        Returns:
            Tensor: 交叉注意力输出，形状与 query 相同，(batch_size, text_len, embed_dim)
        """
        batch_size = query.size(0)  # 获取 batch 大小
        # 线性变换后 reshape 为 (B, num_heads, seq_len, head_dim)，分离多头
        Q = self.q_linear(query).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_linear(key).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_linear(value).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        attention = self._attention(Q, K, V, attention_mask)  # 计算缩放点积注意力
        # 合并多头：(B, heads, text_len, head_dim) → (B, text_len, embed_dim)
        attention = attention.transpose(1, 2).contiguous().view(batch_size, -1, self.embed_dim)
        return self.out(attention)  # 输出线性变换，映射回 embed_dim

    def _attention(self, Q, K, V, attention_mask=None):
        """
        缩放点积注意力核心计算：Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) · V
        Args:
            Q (Tensor): Query，形状 (B, num_heads, text_len, head_dim)
            K (Tensor): Key，形状 (B, num_heads, img_patches, head_dim)
            V (Tensor): Value，形状 (B, num_heads, img_patches, head_dim)
            attention_mask (Tensor, optional): 掩码，形状 (B, text_len)
        Returns:
            Tensor: 加权输出，形状 (B, num_heads, text_len, head_dim)
        """
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)  # 缩放点积注意力分数：(B, heads, text_len, img_patches)
        if attention_mask is not None:                 # 若传入了掩码，则屏蔽 padding 位置的注意力分数
            attention_mask = attention_mask.unsqueeze(1).unsqueeze(1)  # 广播扩展到 (B, 1, 1, seq)
            scores = scores.masked_fill(attention_mask == 0, -1e9)     # padding 位置置为负无穷，softmax 后接近 0
        attention_weights = F.softmax(scores, dim=-1)   # 沿图像 patch 维度归一化为概率分布
        attention = torch.matmul(attention_weights, V)  # 加权求和得到输出：(B, heads, text_len, head_dim)
        return attention  # 交叉注意力加权输出：(B, num_heads, text_len, head_dim)

### 1.5 多模态编码层（MultimodalEncoderLayer）

单层融合结构：**文本自注意力 → 图文交叉注意力 → FFN**，每子层均使用 Pre-LN + 残差连接。

In [ ]:
class MultimodalEncoderLayer(nn.Module):
    """
    多模态编码器单层 = 文本自注意力 + 图文交叉注意力 + 前馈网络（FFN）。
    结构类似 BERT Encoder Layer，但增加了 Cross-Attention 注入图像信息。
    """

    def __init__(self, embed_dim=768, num_heads=12, ff_dim=3072, dropout=0.1):
        """
        初始化多模态编码器单层。
        Args:
            embed_dim (int): 输入/输出特征维度，默认 768
            num_heads (int): 多头注意力头数，默认 12
            ff_dim (int): 前馈网络中间层维度，BERT-base 标准为 3072（4×768），默认 3072
            dropout (float): 随机失活比例，防止过拟合，默认 0.1
        """
        super(MultimodalEncoderLayer, self).__init__()  # 调用父类初始化，注册所有子模块
        self.self_attention = nn.MultiheadAttention(
            embed_dim, num_heads, dropout=dropout, batch_first=True
        )  # 文本内部自注意力（batch_first=True 表示输入形状为 B,S,D）
        self.cross_attention = MultiHeadCrossAttention(embed_dim, num_heads)  # 图文交叉注意力（文本 Query，图像 K/V）
        self.feed_forward = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),   # 升维：768 → 3072
            nn.GELU(),                      # BERT 使用的激活函数（相比 ReLU 更平滑）
            nn.Dropout(dropout),            # FFN 内部 dropout
            nn.Linear(ff_dim, embed_dim),   # 降维回原维度：3072 → 768
            nn.Dropout(dropout),            # FFN 输出 dropout
        )
        self.norm1 = nn.LayerNorm(embed_dim)  # 自注意力前的层归一化（Pre-LN 风格）
        self.norm2 = nn.LayerNorm(embed_dim)  # 交叉注意力前的层归一化
        self.norm3 = nn.LayerNorm(embed_dim)  # FFN 前的层归一化
        self.dropout = nn.Dropout(dropout)    # 残差连接后的 dropout

    def forward(self, text_features, image_features, text_attention_mask=None):
        """
        前向传播：文本特征经过自注意力 + 交叉注意力 + FFN 融合图像信息。
        Args:
            text_features (Tensor): 文本 token 特征，形状 (batch_size, seq_len, embed_dim)
            image_features (Tensor): 图像 patch 特征，形状 (batch_size, num_patches, embed_dim)
            text_attention_mask (Tensor, optional): 文本 padding 掩码，形状 (batch_size, seq_len)
        Returns:
            text_features (Tensor): 融合后的多模态文本特征，形状 (batch_size, seq_len, embed_dim)
        """
        # ---- 子层 1：文本自注意力（Pre-LN + 残差）----
        residual = text_features                          # 保存残差分支
        text_features = self.norm1(text_features)         # Pre-LayerNorm（先归一化再注意力）
        if text_attention_mask is not None:               # 若传入了文本 padding 掩码，则转换为 key_padding_mask
            key_padding_mask = (text_attention_mask == 0)  # True 表示需要被 mask 的 padding 位置
        else:                                             # 未传掩码时设为 None，不做任何 padding 屏蔽
            key_padding_mask = None                       # PyTorch MultiheadAttention 接受 None 表示不屏蔽
        text_attn_out, _ = self.self_attention(
            text_features, text_features, text_features,  # Q=K=V，标准自注意力
            key_padding_mask=key_padding_mask,            # 将 padding 掩码传入，屏蔽无效 token 的注意力权重
        )
        text_features = residual + self.dropout(text_attn_out)  # 残差连接

        # ---- 子层 2：交叉注意力（文本 Query，图像 Key/Value）----
        residual = text_features                          # 保存残差分支
        text_features = self.norm2(text_features)         # Pre-LayerNorm
        cross_attn_out = self.cross_attention(
            query=text_features,    # 文本 token 作为 Query，主动查询图像信息
            key=image_features,     # 图像 patch 作为 Key
            value=image_features,   # 图像 patch 作为 Value
        )  # 交叉注意力输出：(B, seq_len, embed_dim)，文本侧已注入图像信息
        text_features = residual + self.dropout(cross_attn_out)  # 残差连接

        # ---- 子层 3：前馈网络 FFN ----
        residual = text_features                          # 保存残差分支
        text_features = self.norm3(text_features)         # Pre-LayerNorm
        ff_out = self.feed_forward(text_features)         # FFN 升维降维，非线性变换
        text_features = residual + ff_out                 # 残差连接（FFN 子层不加额外 dropout）
        return text_features  # 融合后的多模态文本特征：(batch_size, seq_len, embed_dim)

### 1.6 多模态编码器（MultimodalEncoder）

堆叠多层 `MultimodalEncoderLayer`，对应 ALBEF 论文 Fuse 阶段的 6 层融合编码器。

In [ ]:
class MultimodalEncoder(nn.Module):
    """
    多模态编码器：堆叠多层 MultimodalEncoderLayer，逐层将图像信息融合进文本特征。
    对应 ALBEF 论文中 BERT 后 6 层（此处演示用 6 层）。
    """

    def __init__(self, num_layers=6, embed_dim=768, num_heads=12, ff_dim=3072, dropout=0.1):
        """
        初始化多模态编码器。
        Args:
            num_layers (int): 堆叠的 MultimodalEncoderLayer 层数，论文中为 6
            embed_dim (int): 输入/输出特征维度，需与 BERT 一致，默认 768
            num_heads (int): 多头注意力头数，默认 12
            ff_dim (int): 前馈网络中间层维度，默认 3072（4×embed_dim）
            dropout (float): 随机失活比例，防止过拟合
        """
        super(MultimodalEncoder, self).__init__()  # 调用父类初始化，注册层列表
        self.layers = nn.ModuleList([
            MultimodalEncoderLayer(embed_dim, num_heads, ff_dim, dropout)
            for _ in range(num_layers)  # 创建 num_layers 个融合层，存入 ModuleList 以便自动管理参数
        ])

    def forward(self, text_features, image_features, text_attention_mask=None):
        """
        逐层将图像信息融合进文本特征。
        Args:
            text_features (Tensor): 文本 token 特征，形状 (batch_size, seq_len, embed_dim)
            image_features (Tensor): 图像 patch 特征，形状 (batch_size, num_patches, embed_dim)
            text_attention_mask (Tensor, optional): 文本 padding 掩码，形状 (batch_size, seq_len)；
                                                    1 表示有效 token，0 表示 padding
        Returns:
            text_features (Tensor): 融合后的多模态文本特征，形状 (batch_size, seq_len, embed_dim)；
                                    每个 token 已通过多层 Cross-Attention 看到图像信息
        """
        for layer in self.layers:  # 依次通过每一层 MultimodalEncoderLayer（共 num_layers 层）
            text_features = layer(text_features, image_features, text_attention_mask)  # 逐层注入图像信息
        return text_features  # 最终多模态融合特征：(batch_size, seq_len, embed_dim)

### 1.7 ALBEF 完整模型（ALBEF）

整合图像编码器、文本编码器、多模态编码器及三个任务头，提供统一的前向接口。

In [ ]:
class ALBEF(nn.Module):
    """
    ALBEF 完整模型（教学简化版）。
    三大训练任务：
      - ITC（Image-Text Contrastive）：单模态编码后对比学习，Align 阶段
      - ITM（Image-Text Matching）：多模态融合后判断图文是否匹配
      - MLM（Masked Language Modeling）：多模态融合后预测被 mask 的词
    """

    def __init__(self, embed_dim=768, num_multimodal_layers=6):
        """
        初始化 ALBEF 完整模型，构建所有子模块。
        Args:
            embed_dim (int): 统一的特征维度，所有子模块共享，默认 768
            num_multimodal_layers (int): 多模态融合编码器的堆叠层数，论文为 6
        """
        super(ALBEF, self).__init__()  # 调用父类初始化，注册所有子模块
        self.image_encoder = ImageEncoder(embed_dim)       # 图像单模态编码器（ViT-B/16 骨干，输出 197 个 token）
        self.text_encoder = TextEncoder(embed_dim)         # 文本单模态编码器（BERT-base）
        self.multimodal_encoder = MultimodalEncoder(       # 多模态融合编码器（Fuse 阶段）
            num_layers=num_multimodal_layers,              # 融合层数，论文取 6
            embed_dim=embed_dim,                           # 特征维度，与图像/文本编码器对齐
        )  # 构建 6 层 Cross-Attention 融合编码器
        self.image_proj = nn.Linear(embed_dim, embed_dim)  # ITC：图像对比投影头（映射到对比空间）
        self.text_proj = nn.Linear(embed_dim, embed_dim)   # ITC：文本对比投影头（映射到对比空间）
        self.temperature = nn.Parameter(torch.ones([]) * 0.07)  # 可学习温度参数，缩放相似度分数
        self.itm_head = nn.Linear(embed_dim, 2)            # ITM：二分类头（0=不匹配, 1=匹配）
        self.mlm_head = nn.Linear(embed_dim, 30522)        # MLM：词表预测头，30522 为 BERT vocab 大小

    def forward(self, images, input_ids, attention_mask, task="contrastive", **kwargs):
        """
        统一前向入口，根据 task 参数分发到不同任务。
        Args:
            images (Tensor): 图像输入，形状 (batch_size, 3, H, W)
            input_ids (Tensor): 文本 token ID，形状 (batch_size, seq_len)
            attention_mask (Tensor): 文本 padding 掩码，形状 (batch_size, seq_len)
            task (str): 任务类型，"contrastive" | "itm" | "mlm"
            **kwargs: 透传给子任务的额外参数（如 labels、masked_input_ids 等）
        Returns:
            根据 task 不同，返回对应任务的损失（Tensor 标量）或 logits
        """
        if task == "contrastive":                          # ITC 对比学习任务分支
            return self.contrastive_forward(images, input_ids, attention_mask)  # 返回标量 InfoNCE 损失
        elif task == "itm":                               # ITM 图文匹配任务分支
            return self.itm_forward(images, input_ids, attention_mask, **kwargs)  # 返回二分类损失或 logits
        elif task == "mlm":                               # MLM 掩码语言模型任务分支
            return self.mlm_forward(images, input_ids, attention_mask, **kwargs)  # 返回词表预测损失或 logits
        else:                                             # 传入了不在白名单内的 task 字符串
            raise ValueError(f"不支持的任务类型: {task}")  # 传入非法 task 时抛出异常

    def encode_unimodal(self, images, input_ids, attention_mask):
        """
        单模态编码：图像和文本各自独立编码，不做 Cross-Attention。
        用于 ITC 对比学习（Align Before Fuse 中的 Align 阶段）。
        Args:
            images (Tensor): 图像输入，形状 (batch_size, 3, H, W)
            input_ids (Tensor): 文本 token ID，形状 (batch_size, seq_len)
            attention_mask (Tensor): 文本 padding 掩码，形状 (batch_size, seq_len)
        Returns:
            image_features (Tensor): 图像 token 特征，形状 (batch_size, 197, embed_dim)；197 = [CLS]+196 patch
            text_features (Tensor): 文本 token 特征，形状 (batch_size, seq_len, embed_dim)
        """
        image_features = self.image_encoder(images)                    # 图像编码：(B, 197, 768)；[CLS] 在第 0 位
        text_features = self.text_encoder(input_ids, attention_mask)   # 文本编码：(B, seq, 768)
        return image_features, text_features  # 返回图像特征 (B,197,768) 和文本特征 (B,seq,768)，供后续 ITC 使用

    def encode_multimodal(self, images, input_ids, attention_mask):
        """
        多模态编码：先单模态编码，再经多层 Cross-Attention 融合。
        用于 ITM 和 MLM（Fuse 阶段）。
        Args:
            images (Tensor): 图像输入，形状 (batch_size, 3, H, W)
            input_ids (Tensor): 文本 token ID，形状 (batch_size, seq_len)
            attention_mask (Tensor): 文本 padding 掩码，形状 (batch_size, seq_len)
        Returns:
            multimodal_text_features (Tensor): 多模态融合后文本特征，形状 (batch_size, seq_len, embed_dim)
            image_features (Tensor): 图像 token 特征，形状 (batch_size, 197, embed_dim)；197 = [CLS]+196 patch
        """
        image_features, text_features = self.encode_unimodal(images, input_ids, attention_mask)  # 先各自单模态编码：图像 (B,197,768)，文本 (B,seq,768)
        multimodal_text_features = self.multimodal_encoder(
            text_features, image_features, attention_mask  # 文本查询图像，融合视觉信息
        )  # 多层 Cross-Attention 融合输出：(B, seq_len, embed_dim)
        return multimodal_text_features, image_features  # 返回融合后文本特征 (B,seq,768) 和原始图像特征 (B,197,768)

    def contrastive_forward(self, images, input_ids, attention_mask):
        """
        ITC 图文对比学习（InfoNCE 损失）。
        流程：单模态编码 → 池化全局向量 → 投影 → L2 归一化 → 计算相似度矩阵 → 对称交叉熵
        Args:
            images (Tensor): 图像输入，形状 (batch_size, 3, H, W)
            input_ids (Tensor): 文本 token ID，形状 (batch_size, seq_len)
            attention_mask (Tensor): 文本 padding 掩码
        Returns:
            loss (Tensor): 标量，图文双向 InfoNCE 对比损失的平均值
        """
        image_features, text_features = self.encode_unimodal(images, input_ids, attention_mask)  # 单模态编码：图像 (B,197,768)，文本 (B,seq,768)
        image_embeds = image_features[:, 0, :]        # 图像全局向量：取 ViT [CLS] token（第 0 位），与论文一致
        text_embeds = text_features[:, 0, :]          # 文本全局向量：取 BERT [CLS] token（第 0 位）
        image_embeds = self.image_proj(image_embeds)  # 投影到对比学习空间
        text_embeds = self.text_proj(text_embeds)     # 投影到对比学习空间
        image_embeds = F.normalize(image_embeds, dim=-1)  # L2 归一化，便于用点积算余弦相似度
        text_embeds = F.normalize(text_embeds, dim=-1)    # L2 归一化
        logits = torch.matmul(image_embeds, text_embeds.t()) / self.temperature  # (B,B) 相似度矩阵
        labels = torch.arange(logits.size(0), device=logits.device)  # 对角线为正样本：第 i 张图对应第 i 段文本
        loss_i = F.cross_entropy(logits, labels)      # 图像→文本方向对比损失
        loss_t = F.cross_entropy(logits.t(), labels)  # 文本→图像方向对比损失（对称）
        loss = (loss_i + loss_t) / 2                  # 双向对称损失取平均
        return loss  # 标量 InfoNCE 对比损失

    def itm_forward(self, images, input_ids, attention_mask, labels=None):
        """
        ITM 图文匹配：判断图文是否匹配（二分类）。
        Args:
            images (Tensor): 图像输入，形状 (batch_size, 3, H, W)
            input_ids (Tensor): 文本 token ID，形状 (batch_size, seq_len)
            attention_mask (Tensor): 文本 padding 掩码
            labels (Tensor, optional): 形状 (batch_size,)，1=匹配, 0=不匹配；为 None 时只返回 logits
        Returns:
            loss (Tensor): 有 labels 时返回标量二分类交叉熵损失
            itm_logits (Tensor): 无 labels 时返回形状 (batch_size, 2) 的分类 logits
        """
        multimodal_features, _ = self.encode_multimodal(images, input_ids, attention_mask)  # 多模态融合编码；图像特征用 _ 丢弃，只保留文本侧融合结果 (B,seq,768)
        cls_features = multimodal_features[:, 0, :]  # 取融合后 [CLS] 向量作为句级表示：(B, 768)
        itm_logits = self.itm_head(cls_features)     # 线性分类头：(B, 2)
        if labels is not None:                           # 训练模式：提供了真实标签
            loss = F.cross_entropy(itm_logits, labels)  # 二分类损失
            return loss                                  # 返回标量二分类交叉熵损失
        else:                                            # 推理模式：未提供标签
            return itm_logits  # 推理时直接返回 logits

    def mlm_forward(self, images, input_ids, attention_mask, masked_input_ids=None, mlm_labels=None):
        """
        MLM 掩码语言模型：随机 mask 部分词，结合图像信息预测原词。
        Args:
            images (Tensor): 图像输入，形状 (batch_size, 3, H, W)
            input_ids (Tensor): 原始文本 token ID，形状 (batch_size, seq_len)
            attention_mask (Tensor): 文本 padding 掩码
            masked_input_ids (Tensor, optional): 含 [MASK] 的 token ID；为 None 时自动生成
            mlm_labels (Tensor, optional): 被 mask 位置的原始 token ID，其余为 -100
        Returns:
            loss (Tensor): 有 labels 时返回标量词表预测交叉熵损失
            mlm_logits (Tensor): 无 labels 时返回形状 (B, seq_len, vocab_size) 的 logits
        """
        if masked_input_ids is None and mlm_labels is None:  # 未传入预处理好的 mask 样本时，自动生成
            masked_input_ids, mlm_labels = self.mask_tokens(input_ids)  # 自动生成 BERT 风格的 mask
        multimodal_features, _ = self.encode_multimodal(images, masked_input_ids, attention_mask)  # 用含 [MASK] 的序列做多模态融合编码，返回 (B,seq,768)
        mlm_logits = self.mlm_head(multimodal_features)  # 每个 token 位置预测词表概率：(B, seq, 30522)
        if mlm_labels is not None:                        # 训练模式：提供了真实标签
            loss = F.cross_entropy(
                mlm_logits.view(-1, mlm_logits.size(-1)),  # 展平为 (B*seq, vocab_size)
                mlm_labels.view(-1),                       # 展平为 (B*seq,)
                ignore_index=-100,                         # 未 mask 的位置标签为 -100，不参与损失
            )
            return loss                                   # 返回标量词表预测交叉熵损失
        else:                                             # 推理模式：未提供标签
            return mlm_logits  # 推理时直接返回各 token 位置的词表 logits：(B, seq_len, 30522)

    def mask_tokens(self, input_ids, mlm_probability=0.15):
        """
        按 BERT 规则创建 MLM 训练样本。
        15% token 被选中；其中 80% 换 [MASK]，10% 换随机词，10% 保持不变。
        Args:
            input_ids (Tensor): 原始 token ID，形状 (batch_size, seq_len)
            mlm_probability (float): 被 mask 的 token 比例，BERT 标准为 0.15
        Returns:
            input_ids (Tensor): 含 [MASK] 的 token ID，形状 (batch_size, seq_len)
            labels (Tensor): 被 mask 位置的原始 ID，其余为 -100，形状 (batch_size, seq_len)
        """
        device = input_ids.device                  # 与输入在同一设备
        labels = input_ids.clone()                 # 复制一份作为预测标签，后续在非 mask 位置写 -100
        probability_matrix = torch.full(labels.shape, mlm_probability, device=device)  # 全表初始化为 mask 概率
        special_tokens_mask = torch.zeros_like(input_ids, dtype=torch.bool, device=device)  # 特殊 token 位置标记矩阵，初始全 False
        for special_id in [0, 101, 102]:           # PAD=0, [CLS]=101, [SEP]=102，不参与 mask
            special_tokens_mask = special_tokens_mask | (input_ids == special_id)  # 逐一标记特殊 token 位置为 True
        probability_matrix.masked_fill_(special_tokens_mask, value=0.0)  # 特殊 token 的 mask 概率置 0
        masked_indices = torch.bernoulli(probability_matrix).bool()       # 按概率采样待 mask 位置
        labels[~masked_indices] = -100             # 非 mask 位置标签设为 -100（损失中忽略）
        indices_replaced = torch.bernoulli(torch.full(labels.shape, 0.8, device=device)).bool() & masked_indices  # 80% 的被选中 token：替换为 [MASK]
        input_ids[indices_replaced] = 103          # 80% 的 mask 位置替换为 [MASK]（token ID=103）
        indices_random = torch.bernoulli(torch.full(labels.shape, 0.5, device=device)).bool() & masked_indices & ~indices_replaced  # 剩余 20% 中再取 50%：替换为随机词
        random_words = torch.randint(0, 30522, labels.shape, device=device)  # 随机词 ID（vocab_size=30522）
        input_ids[indices_random] = random_words[indices_random]             # 10% 的位置替换为随机词
        return input_ids, labels  # 返回含 [MASK] 的 token ID 序列 和 对应标签（非 mask 位置为 -100）

### 1.8 三任务演示（ITC / ITM / MLM）

实例化完整 ALBEF 模型，依次调用三种训练任务的前向传播，并打印特征形状与损失值。

In [ ]:
def demo_albef_with_cross_attention():
    """
    演示函数：依次展示单模态编码、多模态融合、ITC/ITM/MLM 三个任务。
    使用随机图像 + 4 条英文句子，无需下载数据集。
    """
    print("=" * 80)                              # 打印顶部分隔线，美化输出
    print("ALBEF模型 - 包含交叉注意力机制")        # 演示标题
    print("=" * 80)                              # 打印分隔线

    model = ALBEF(embed_dim=768, num_multimodal_layers=6)  # 实例化完整 ALBEF 模型
    model.eval()  # 评估模式：关闭 dropout，固定 BN 统计量

    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')  # 加载 BERT 分词器
    texts = [                                    # 4 条英文演示句子，与 4 张随机图像一一对应
        "A cat sitting on a chair",
        "A dog running in the park",
        "A bird flying in the sky",
        "A fish swimming in water",
    ]
    tokenized = tokenizer(
        texts, padding=True, truncation=True, return_tensors='pt', max_length=32
    )  # 转 token ID，padding 到同一长度，返回 PyTorch 张量
    images = torch.randn(4, 3, 224, 224)  # 4 张随机 RGB 图像（演示用，非真实图片）

    print("输入数据形状:")                        # 打印输入数据维度信息
    print(f"  图像: {images.shape}")           # (4, 3, 224, 224)
    print(f"  文本ID: {tokenized['input_ids'].shape}")         # (4, seq_len)
    print(f"  注意力掩码: {tokenized['attention_mask'].shape}")  # (4, seq_len)
    print()                                      # 空行，分隔各输出块

    with torch.no_grad():  # 演示推理不需要梯度，节省显存
        print("1. 单模态编码")                   # 单模态编码阶段标题
        print("-" * 40)                          # 打印小节分隔线
        image_features, text_features = model.encode_unimodal(
            images, tokenized['input_ids'], tokenized['attention_mask']
        )  # 图像→(B,197,768)（[CLS]+196 patch token），文本→(B,seq,768)；各自独立编码，不做 Cross-Attention
        print(f"  图像特征形状: {image_features.shape}")   # (4, 197, 768)
        print(f"  文本特征形状: {text_features.shape}")    # (4, seq_len, 768)
        print()                                  # 空行，分隔各任务输出

        print("2. 多模态编码（交叉注意力）")       # 多模态融合阶段标题
        print("-" * 40)                          # 打印小节分隔线
        multimodal_features, _ = model.encode_multimodal(
            images, tokenized['input_ids'], tokenized['attention_mask']
        )  # 先单模态编码，再经多层 Cross-Attention 融合，返回 (B, seq_len, 768)
        print(f"  多模态特征形状: {multimodal_features.shape}")  # (4, seq_len, 768)
        print("  文本特征已通过交叉注意力与图像特征融合")  # 说明融合完成
        print()                                  # 空行

        print("3. 对比学习任务 (ITC)")            # ITC 任务标题
        print("-" * 40)                          # 打印小节分隔线
        contrastive_loss = model(
            images, tokenized['input_ids'], tokenized['attention_mask'], task="contrastive"
        )  # ITC 前向：单模态编码 → 投影 → L2 归一化 → 双向 InfoNCE 对比损失
        print(f"  对比学习损失: {contrastive_loss.item():.4f}")  # InfoNCE 标量损失
        print()                                  # 空行

        print("4. 图像-文本匹配任务 (ITM)")        # ITM 任务标题
        print("-" * 40)                          # 打印小节分隔线
        itm_labels = torch.tensor([1, 1, 0, 0])  # 前 2 对匹配，后 2 对不匹配
        itm_loss = model(
            images, tokenized['input_ids'], tokenized['attention_mask'],
            task="itm", labels=itm_labels,
        )  # ITM 前向：多模态融合后取 [CLS] 向量，二分类交叉熵损失
        print(f"  ITM损失: {itm_loss.item():.4f}")  # 二分类交叉熵损失
        itm_logits = model.itm_forward(images, tokenized['input_ids'], tokenized['attention_mask'])  # 推理模式（无 labels）：返回 logits，形状 (4, 2)
        itm_predictions = torch.softmax(itm_logits, dim=-1)  # 转为概率，形状 (4, 2)
        print("  ITM预测概率:")                    # 打印每个样本的匹配概率
        for i, pred in enumerate(itm_predictions):  # 遍历 batch 中每个样本的预测结果
            print(f"    样本{i+1}: 不匹配={pred[0]:.3f}, 匹配={pred[1]:.3f}")  # pred[0]=不匹配, pred[1]=匹配
        print()                                  # 空行

        print("5. 掩码语言模型任务 (MLM)")         # MLM 任务标题
        print("-" * 40)                          # 打印小节分隔线
        mlm_loss = model(images, tokenized['input_ids'], tokenized['attention_mask'], task="mlm")  # MLM 前向：自动生成 mask，结合图像预测被 mask 词，返回标量损失
        print(f"  MLM损失: {mlm_loss.item():.4f}")  # 词表预测交叉熵损失
        print("  MLM预测基于图像-文本交叉注意力特征")  # 说明 MLM 使用多模态融合特征
        print()                                  # 空行

        print("6. 模型架构特点")                  # 模型统计信息标题
        print("-" * 40)                          # 打印小节分隔线
        total_params = sum(p.numel() for p in model.parameters())  # 统计全部可学习参数量
        multimodal_params = sum(p.numel() for p in model.multimodal_encoder.parameters())  # 仅多模态编码器参数量
        print(f"  总参数量: {total_params:,}")                              # 打印总参数量（逗号分隔便于阅读）
        print(f"  多模态编码器参数: {multimodal_params:,}")                 # 打印多模态编码器参数量
        print(f"  交叉注意力层数: {len(model.multimodal_encoder.layers)}")  # 打印 Cross-Attention 堆叠层数
        print("  包含图像-文本交叉注意力机制")      # 说明架构包含跨模态注意力
        print("  符合ALBEF论文架构")               # 与论文对齐说明

    print("=" * 80)                              # 打印底部分隔线
    print("演示完成！")                           # 演示结束提示
    print("=" * 80)                              # 打印分隔线


if __name__ == "__main__":                       # 直接运行脚本时执行（Jupyter 中不生效，保留工程规范）
    torch.manual_seed(42)    # 固定 PyTorch 随机种子，结果可复现
    np.random.seed(42)       # 固定 NumPy 随机种子
    demo_albef_with_cross_attention()  # 运行完整演示

## 二、动量编码器 + 负样本队列 + 动量蒸馏（ITC 增强）

本节在基础架构之上，为 ITC 对比学习增加以下三项机制（对应 ALBEF 论文 3.3-3.5 节）：

| 机制 | 作用 | 类比 |
|------|------|------|
| **动量编码器（Momentum Encoder）** | EMA 缓慢更新的 Teacher，提供稳定的特征 | MoCo 中的 key encoder |
| **负样本队列（Feature Queue）** | 存储历史动量特征，扩充 ITC 负样本池 | MoCo 中的 queue |
| **动量蒸馏（Momentum Distillation）** | KL 散度让 Student 学习 Teacher 的软概率分布 | 知识蒸馏 |

> 注意：动量机制**仅用于 ITC**，ITM 和 MLM 仍使用 Student 主模型前向传播。

### 2.1 特征队列（FeatureQueue）与动量更新函数（momentum_update）

`FeatureQueue`：环形 FIFO 缓冲区，存储历史 Teacher 特征作为额外负样本。  
`momentum_update`：EMA 更新函数，将 Student 参数缓慢同步到 Teacher。

In [ ]:
# ============================================================
# ALBEF 教学版 Notebook - 特征队列与动量更新
# 内容：FeatureQueue（负样本存储）+ momentum_update（EMA 更新）
# 依赖：需先运行依赖导入 + 各模型类 cell
# ============================================================

import copy  # 深拷贝动量编码器参数（Teacher 初始与 Student 相同）


class FeatureQueue:
    """
    FIFO 特征队列：存放动量编码器输出的历史特征，作为 ITC 额外负样本。
    思路来自 MoCo；论文队列大小 65536，教学演示默认 1024。
    """

    def __init__(self, queue_size=1024, embed_dim=768):
        """
        初始化特征队列（仅记录参数，队列 tensor 延迟到 _init_queue 中分配）。
        Args:
            queue_size (int): 队列最大容量（特征条数），论文为 65536，演示默认 1024
            embed_dim (int): 每条特征的维度（与 ITC 投影后维度一致），默认 768
        """
        self.queue_size = queue_size  # 队列容量
        self.embed_dim = embed_dim    # 特征维度

    def _init_queue(self, device):
        """
        首次使用时，在指定设备（CPU/GPU）上分配队列张量。
        Args:
            device (torch.device): 目标设备，与模型输入保持一致
        """
        self.queue = torch.zeros(self.queue_size, self.embed_dim, device=device)  # 预分配内存：(queue_size, embed_dim)
        self.ptr = 0          # 写入指针（下一个写入位置）
        self.is_full = False  # 队列是否已被写满过（用于环形覆盖判断）

    def enqueue(self, features):
        """
        将当前 batch 的动量特征写入队列（环形缓冲区，detach 避免梯度回传）。
        Args:
            features (Tensor): 形状 (batch_size, embed_dim)，已 L2 归一化的动量特征
        """
        batch_size = features.shape[0]                            # 当前 batch 大小
        if self.ptr + batch_size <= self.queue_size:              # 判断剩余空间是否足够容纳当前 batch
            # 剩余空间足够：顺序写入
            self.queue[self.ptr:self.ptr + batch_size] = features.detach()  # detach 避免梯度回传
            self.ptr += batch_size                                # 指针右移
        else:
            # 空间不足：尾部写满后从头覆盖（环形队列）
            remain = self.queue_size - self.ptr                   # 尾部剩余空间
            self.queue[self.ptr:] = features[:remain].detach()   # 写入尾部
            self.queue[:batch_size - remain] = features[remain:].detach()  # 从头部继续写
            self.ptr = batch_size - remain                        # 更新指针位置
            self.is_full = True                                   # 标记队列已满

    def get_queue(self):
        """
        返回当前队列中所有有效特征，供 ITC 计算负样本相似度。
        Returns:
            Tensor: 有效特征张量，形状 (valid_size, embed_dim)；
                    valid_size：队列未满时为 ptr（当前写入位置），满后等于 queue_size
        """
        if self.is_full:                # 队列已至少被写满一次，所有位置均有效
            return self.queue           # 队列已满，返回全部 queue_size 条特征
        return self.queue[:self.ptr]    # 未满，只返回 ptr 之前已写入的部分


@torch.no_grad()  # EMA 更新不需要梯度
def momentum_update(student_module, momentum_module, momentum=0.995):
    """
    指数移动平均（EMA）更新动量编码器（Teacher）参数。
    公式：θ_m ← λ·θ_m + (1-λ)·θ_s
    Args:
        student_module (nn.Module): Student 子模块（参与梯度更新）
        momentum_module (nn.Module): Momentum 子模块（缓慢跟随 Student）
        momentum (float): λ，EMA 系数，论文取 0.995；越大 Teacher 越稳定
    """
    for param_s, param_m in zip(student_module.parameters(), momentum_module.parameters()):  # 同步遍历 Student 与 Teacher 的对应参数
        param_m.data = param_m.data * momentum + param_s.data * (1.0 - momentum)  # EMA 公式更新 Teacher 参数


### 2.2 带动量的 ALBEF（ALBEFWithMomentum）

在 Student-Teacher 框架下，整合负样本队列与动量蒸馏损失，实现 ALBEF 论文 3.3–3.5 节的完整 ITC 增强机制。

In [ ]:
class ALBEFWithMomentum(nn.Module):
    """
    在 ALBEF 基础上为 ITC 增加：
      1. 动量编码器（Teacher，EMA 更新，参数缓慢变化）
      2. 图文双队列（存历史 Teacher 特征，扩充 ITC 负样本）
      3. 动量蒸馏（KL 散度，让 Student 学习 Teacher 的软标签，缓解噪声图文对）
    注意：动量机制仅用于 ITC，ITM/MLM 仍走 Student 主模型。
    相对复杂，可以不掌握。
    """

    def __init__(self, base_model=None, queue_size=1024, momentum=0.995, embed_dim=768):
        """
        初始化带动量机制的 ALBEF 模型。
        Args:
            base_model (ALBEF, optional): 预构建的 Student 模型；为 None 时自动创建新 ALBEF 实例
            queue_size (int): 负样本特征队列容量，论文为 65536，教学演示默认 1024
            momentum (float): EMA 动量系数 λ，越大 Teacher 更新越慢、特征越稳定，论文取 0.995
            embed_dim (int): 特征维度，须与 Student 模型保持一致，默认 768
        """
        super().__init__()  # 调用父类 nn.Module 初始化，注册所有子模块
        self.student = base_model if base_model is not None else ALBEF(embed_dim=embed_dim)  # Student 主模型（有梯度，参与训练）
        self.momentum = momentum      # EMA 系数 λ，控制 Teacher 更新速度
        self.queue_size = queue_size  # 负样本队列容量

        # 深拷贝 ITC 相关模块作为动量编码器（Teacher 初始与 Student 完全相同）
        self.momentum_image_encoder = copy.deepcopy(self.student.image_encoder)  # Teacher 图像编码器
        self.momentum_text_encoder = copy.deepcopy(self.student.text_encoder)    # Teacher 文本编码器
        self.momentum_image_proj = copy.deepcopy(self.student.image_proj)        # Teacher 图像投影头
        self.momentum_text_proj = copy.deepcopy(self.student.text_proj)          # Teacher 文本投影头

        # Teacher 不参与反向传播，冻结参数以节省显存
        for module in [                              # 遍历所有 Teacher 子模块
            self.momentum_image_encoder,
            self.momentum_text_encoder,
            self.momentum_image_proj,
            self.momentum_text_proj,
        ]:
            for param in module.parameters():        # 遍历该模块的所有参数张量
                param.requires_grad = False          # 冻结 Teacher 参数，仅通过 EMA 更新

        self.image_queue = FeatureQueue(queue_size=queue_size, embed_dim=embed_dim)  # 图像特征 FIFO 队列（负样本池）
        self.text_queue = FeatureQueue(queue_size=queue_size, embed_dim=embed_dim)   # 文本特征 FIFO 队列（负样本池）

    def _init_queues(self, device):
        """
        将队列张量初始化到与输入相同的设备上。
        Args:
            device (torch.device): 目标设备（CPU 或 GPU），与模型输入保持一致
        """
        self.image_queue._init_queue(device)  # 初始化图像队列 tensor 到指定设备
        self.text_queue._init_queue(device)   # 初始化文本队列 tensor 到指定设备

    def encode_itc(self, images, input_ids, attention_mask, use_momentum=False):
        """
        提取 ITC 用的 L2 归一化全局向量（可选 Student 或 Teacher）。
        Args:
            images (Tensor): 图像输入，形状 (batch_size, 3, H, W)
            input_ids (Tensor): 文本 token ID，形状 (batch_size, seq_len)
            attention_mask (Tensor): 文本 padding 掩码，形状 (batch_size, seq_len)
            use_momentum (bool): True 使用 Teacher（动量编码器），False 使用 Student
        Returns:
            image_embeds (Tensor): 图像全局向量（L2 归一化），形状 (batch_size, embed_dim)
            text_embeds (Tensor): 文本全局向量（L2 归一化），形状 (batch_size, embed_dim)
        """
        if use_momentum:                                     # 如果使用动量编码器（Teacher）
            image_encoder = self.momentum_image_encoder     # 用动量版的图像编码器
            text_encoder = self.momentum_text_encoder       # 用动量版的文本编码器
            image_proj = self.momentum_image_proj           # 用动量版的图像投影头
            text_proj = self.momentum_text_proj             # 用动量版的文本投影头
        else:                                              # 使用 Student 主模型
            image_encoder = self.student.image_encoder      # 用 Student 的图像编码器
            text_encoder = self.student.text_encoder        # 用 Student 的文本编码器
            image_proj = self.student.image_proj            # 用 Student 的图像投影头
            text_proj = self.student.text_proj              # 用 Student 的文本投影头

        image_features = image_encoder(images)                                         # 图像编码：(B, 197, 768)；[CLS] 在第 0 位
        text_features = text_encoder(input_ids, attention_mask)                       # 文本编码：(B, seq, 768)
        image_embeds = F.normalize(image_proj(image_features[:, 0, :]), dim=-1)      # 取 ViT [CLS] token + 投影 + L2 归一化：(B, 768)
        text_embeds = F.normalize(text_proj(text_features[:, 0, :]), dim=-1)         # 取 BERT [CLS] token + 投影 + L2 归一化：(B, 768)
        return image_embeds, text_embeds  # 返回 L2 归一化后的图像向量和文本向量，形状均为 (B, embed_dim)

    def contrastive_forward_with_momentum(self, images, input_ids, attention_mask, alpha=0.4):
        """
        带队列与动量蒸馏的 ITC 损失。
        total_loss = (1-α)·InfoNCE + α·KL(Student || Teacher)
        Args:
            images (Tensor): 图像输入，形状 (batch_size, 3, H, W)
            input_ids (Tensor): 文本 token ID，形状 (batch_size, seq_len)
            attention_mask (Tensor): 文本 padding 掩码，形状 (batch_size, seq_len)
            alpha (float): 蒸馏损失权重，论文取 0.4
        Returns:
            dict: 损失字典，键包含 total_loss / contrastive_loss / distill_loss / queue_text_size / queue_image_size
        """
        device = images.device                                                # 当前设备（CPU/GPU）
        if not hasattr(self.image_queue, 'queue'):                            # 若队列未初始化（首次调用）
            self._init_queues(device)                                         # 将队列 tensor 放到正确设备

        batch_size = images.size(0)                                           # 当前 batch 大小
        temperature = self.student.temperature                                # InfoNCE 温度超参（可学习）

        # ---- Student 特征（有梯度，参与优化）----
        image_embeds_s, text_embeds_s = self.encode_itc(
            images, input_ids, attention_mask, use_momentum=False             # 提取 Student 图文全局特征
        )

        # ---- Momentum（Teacher）特征（无梯度）+ 写入队列 ----
        with torch.no_grad():                                                 # Teacher 前向与入队不开梯度
            image_embeds_m, text_embeds_m = self.encode_itc(
                images, input_ids, attention_mask, use_momentum=True          # 提取 Teacher 图文全局特征（软标签来源）
            )
            self.image_queue.enqueue(image_embeds_m)                          # 图像 Teacher 特征入队，扩充负样本
            self.text_queue.enqueue(text_embeds_m)                            # 文本 Teacher 特征入队，扩充负样本

        queue_images = self.image_queue.get_queue()                           # 历史图像 Teacher 特征：(Q, embed_dim)
        queue_texts = self.text_queue.get_queue()                             # 历史文本 Teacher 特征：(Q, embed_dim)

        # ---- 图像→文本 InfoNCE：正样本=batch 内配对，负样本=队列 ----
        pos_i2t = torch.matmul(image_embeds_s, text_embeds_m.T) / temperature    # Student 图像 VS batch 内 Teacher 文本（含正例）
        neg_i2t = torch.matmul(image_embeds_s, queue_texts.T) / temperature if queue_texts.numel() > 0 else None  # Student 图像 VS 队列历史文本（纯负例）
        if neg_i2t is not None and neg_i2t.shape[1] > 0:  # 队列不为空时，将 batch 内正负样本与队列负样本拼接
            logits_i2t = torch.cat([pos_i2t, neg_i2t], dim=1)                    # 合并：(B, B+Q)
        else:                                                                      # 队列尚空（初始几步）
            logits_i2t = pos_i2t                                                 # 队列为空时只用 batch 内样本
        labels_i2t = torch.arange(batch_size, device=device)                     # 正样本位置：对角线（第 i 图 ↔ 第 i 文本）
        loss_i2t = F.cross_entropy(logits_i2t, labels_i2t)                      # 图像→文本对比损失

        # ---- 文本→图像 InfoNCE（对称方向）----
        pos_t2i = torch.matmul(text_embeds_s, image_embeds_m.T) / temperature    # Student 文本 VS batch 内 Teacher 图像（含正例）
        neg_t2i = torch.matmul(text_embeds_s, queue_images.T) / temperature if queue_images.numel() > 0 else None  # Student 文本 VS 队列历史图像（纯负例）
        if neg_t2i is not None and neg_t2i.shape[1] > 0:  # 队列不为空，拼接 batch 内和队列负样本
            logits_t2i = torch.cat([pos_t2i, neg_t2i], dim=1)                   # 合并：(B, B+Q)
        else:                                                                      # 队列尚空（初始几步）
            logits_t2i = pos_t2i                                                 # 队列为空时只用 batch 内样本
        labels_t2i = torch.arange(batch_size, device=device)                     # 正样本 label：对角线
        loss_t2i = F.cross_entropy(logits_t2i, labels_t2i)                      # 文本→图像对比损失
        # batch 内：另外 B-1 条文本是「同场考生」里的干扰项；
        # 队列里：Q 条历史文本是「从往届题库抽来的」干扰项；
        # 合在一起，每张图的负样本从 B-1 变成 (B-1)+Q，对比学习更难、也更有效
        contrastive_loss = (loss_i2t + loss_t2i) / 2                            # 双向对称对比损失取平均

        # ---- 动量蒸馏：让 Student 的软分布接近 Teacher ----
        with torch.no_grad():  # 计算 Teacher 软标签不需要梯度
            teacher_logits = torch.matmul(image_embeds_m, text_embeds_m.T) / temperature  # Teacher 的相似度 logits（无梯度）
            teacher_probs = F.softmax(teacher_logits, dim=-1)                             # Teacher 的概率分布（软标签）
        student_logits = torch.matmul(image_embeds_s, text_embeds_m.detach().T) / temperature  # Student 的 logits，text_embeds_m 停止梯度
        student_log_probs = F.log_softmax(student_logits, dim=-1)                             # Student 的对数概率（KL 需要对数形式）
        distill_loss = F.kl_div(student_log_probs, teacher_probs, reduction='batchmean')      # KL(Teacher || Student)，Student 向 Teacher 靠拢

        total_loss = (1 - alpha) * contrastive_loss + alpha * distill_loss                   # 加权总损失

        return {
            'total_loss': total_loss,                   # 总损失（用于 backward）
            'contrastive_loss': contrastive_loss,       # 单纯 InfoNCE 对比损失
            'distill_loss': distill_loss,               # 动量蒸馏 KL 损失
            'queue_text_size': queue_texts.shape[0] if queue_texts.numel() > 0 else 0,     # 文本队列当前有效特征数
            'queue_image_size': queue_images.shape[0] if queue_images.numel() > 0 else 0,  # 图像队列当前有效特征数
        }

    def update_momentum_encoder(self):
        """
        每个 optimizer.step() 之后调用，用 EMA 同步 Teacher 参数。
        对图像编码器、文本编码器、图像投影头、文本投影头分别做 EMA 更新。
        """
        momentum_update(self.student.image_encoder, self.momentum_image_encoder, self.momentum)  # EMA 同步图像编码器
        momentum_update(self.student.text_encoder, self.momentum_text_encoder, self.momentum)    # EMA 同步文本编码器
        momentum_update(self.student.image_proj, self.momentum_image_proj, self.momentum)        # EMA 同步图像投影头
        momentum_update(self.student.text_proj, self.momentum_text_proj, self.momentum)          # EMA 同步文本投影头

    def forward(self, images, input_ids, attention_mask, task="contrastive_momentum", **kwargs):
        """
        统一前向入口，根据 task 参数分发任务。
        Args:
            images (Tensor): 图像输入，形状 (batch_size, 3, H, W)
            input_ids (Tensor): 文本 token ID，形状 (batch_size, seq_len)
            attention_mask (Tensor): 文本 padding 掩码，形状 (batch_size, seq_len)
            task (str): 任务类型；"contrastive_momentum" 执行带动量蒸馏的 ITC，其余转发至 Student
            **kwargs: 透传给子任务的额外参数（如 alpha 蒸馏权重、itm labels 等）
        Returns:
            带动量蒸馏时返回损失字典（含 total_loss / contrastive_loss / distill_loss / queue_size）；
            其他任务返回 Student 模型的对应输出（损失标量或 logits）
        """
        if task == "contrastive_momentum":         # 带动量蒸馏的 ITC 任务分支
            return self.contrastive_forward_with_momentum(
                images, input_ids, attention_mask, alpha=kwargs.get('alpha', 0.4)  # 默认 alpha=0.4
            )  # 返回包含 total_loss / contrastive_loss / distill_loss 的损失字典
        return self.student(images, input_ids, attention_mask, task=task, **kwargs)  # 其余任务（ITM/MLM）直接转发给 Student 主模型

### 2.3 演示：动量蒸馏短程训练

运行 5 步训练，逐步观察负样本队列扩充过程及对比损失、蒸馏损失的变化趋势。

In [ ]:
def demo_momentum_distillation(queue_size=1024, train_steps=5, alpha=0.4):
    """
    短程训练演示：观察队列增长与各损失变化，验证动量蒸馏机制。
    可在普通 GPU/CPU 运行，无需 A100。
    Args:
        queue_size (int): 负样本队列大小，演示默认 1024
        train_steps (int): 训练步数，演示默认 5 步
        alpha (float): 蒸馏损失权重，演示默认 0.4
    """
    print("=" * 80)                              # 打印顶部分隔线，美化输出
    print("ALBEF 轻量版：动量蒸馏 + 负样本队列（教学演示）")  # 演示标题
    print("=" * 80)                              # 打印分隔线
    print(f"配置: queue_size={queue_size}, train_steps={train_steps}, alpha={alpha}, momentum=0.995")  # 打印关键超参数
    print("说明: 论文队列 65536 + 8×A100；此处为小规模演示，原理一致")  # 与论文规模对比说明
    print()                                      # 空行，分隔输出块

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # 自动选择 GPU 或 CPU
    print(f"运行设备: {device}")                  # 打印当前使用的计算设备
    print()                                      # 空行

    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')  # 加载 BERT 分词器
    texts = [                                    # 4 条英文演示句子，与 4 张随机图像一一对应
        "A cat sitting on a chair",
        "A dog running in the park",
        "A bird flying in the sky",
        "A fish swimming in water",
    ]
    tokenized = tokenizer(texts, padding=True, truncation=True, return_tensors='pt', max_length=32)  # 分词并 padding 到相同长度
    images = torch.randn(4, 3, 224, 224, device=device)            # 4 张随机 RGB 图像（演示用）
    input_ids = tokenized['input_ids'].to(device)                  # 将 token ID 搬到目标设备
    attention_mask = tokenized['attention_mask'].to(device)        # 将注意力掩码搬到目标设备

    model = ALBEFWithMomentum(queue_size=queue_size, momentum=0.995).to(device)  # 实例化带动量的 ALBEF 并移到设备
    model.train()  # 训练模式：启用 dropout 等

    optimizer = torch.optim.AdamW(
        [p for p in model.student.parameters() if p.requires_grad],  # 只优化 Student 参数（Teacher 冻结）
        lr=1e-5,
    )

    print("开始短程训练演示（仅 ITC + 动量蒸馏）...")  # 提示训练开始
    print("-" * 60)                              # 打印分隔线
    for step in range(1, train_steps + 1):       # 逐步训练，共 train_steps 步
        optimizer.zero_grad()  # 清空上一步梯度，防止梯度累积
        outputs = model(images, input_ids, attention_mask, task="contrastive_momentum", alpha=alpha)  # 带动量蒸馏的 ITC 前向
        outputs['total_loss'].backward()  # 反向传播，计算所有 Student 参数的梯度
        optimizer.step()                  # 用 AdamW 更新 Student 参数
        model.update_momentum_encoder()   # EMA 更新 Teacher 参数

        print(
            f"Step {step:02d} | total={outputs['total_loss'].item():.4f} "
            f"| contrastive={outputs['contrastive_loss'].item():.4f} "
            f"| distill={outputs['distill_loss'].item():.4f} "
            f"| text_queue={outputs['queue_text_size']} "   # 文本队列当前有效特征数
            f"| image_queue={outputs['queue_image_size']}"  # 图像队列当前有效特征数
        )

    print("-" * 60)                              # 打印结尾分隔线
    print("演示完成！")                           # 演示结束提示
    print("- 动量编码器: EMA 缓慢跟随 Student，提供稳定 Teacher 特征")  # 动量编码器功能说明
    print("- 负样本队列: 存放历史动量特征，扩充 ITC 负样本")           # 负样本队列功能说明
    print("- 动量蒸馏: KL 让 Student 学习 Teacher 的软概率分布")       # 动量蒸馏功能说明
    print("=" * 80)                              # 打印底部分隔线


# 运行轻量演示（约 1~3 分钟，普通 GPU/CPU 均可）
demo_momentum_distillation(queue_size=1024, train_steps=5, alpha=0.4)

## 补充说明

- `transformers` 库中**没有** ALBEF 官方实现，本 Notebook 为手写教学版。
- **Cell 1**：基础架构 + ITC / ITM / MLM 三任务演示（每行均有详细中文注释）。
- **Cell 2**：轻量版动量蒸馏 + 负样本队列（`queue_size=1024`，普通 GPU/CPU 可运行）。
- **建议学习顺序**：先通读 Cell 1 注释理解架构 → 运行演示 → 再学 Cell 2 动量机制。
